# WFS Frame Denoiser

A small FiLM-conditioned encoder-decoder CNN that learns to map the WFS's noisy raw detector frame(s) back toward the noiseless current frame. `WFS.Propagator` (`AI4AO/TorchPropagator.py`) already computes a noiseless frame (`wfs.frame_no_noise`) on the way to the noisy one it actually returns (`wfs.frame_with_noise`, via `AddNoiseToFrame`) whenever `useNoise=True` -- both are available after every forward call, so `frame_no_noise` can be used directly as a supervised training target, with no separate ground-truth acquisition needed.

Unlike most other notebooks in `Tutorials/Advanced/`, this one does **not** lean on the simulator's differentiability the way a reconstructor-training notebook does -- the WFS/DM twin is used purely as a frozen, no-grad data generator here. The only thing being trained is the denoiser itself, and the real question this notebook answers isn't "does the frame look cleaner" but **does denoising actually improve wavefront reconstruction** -- validated at the end via the classical `GetReconstructedOPD` path, comparing residual wavefront error (nm RMS) from raw-noisy vs. denoised vs. oracle-noiseless frames across a range of photon counts.

The denoiser can also see a short history of past frames, not just the current one: `PhaseDataset`'s wind-translated atmosphere means consecutive frames share a correlated underlying wavefront while their photon/read noise is independent, so stacking `n_frames` consecutive frames gives the network multiple independent noisy looks at a (nearly) shared signal -- similar in spirit to burst-mode denoising. This notebook trains and compares `n_frames = 1, 5, 10`.

Full design rationale: `Ideas/12-wfs-frame-denoiser.md`.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import os

from AI4AO import PyramidWFS, PhaseDataset, DeformableMirror, imshow_multiple

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Configuration and instrument

Same fictional-demo, nominal-geometry modulated Pyramid WFS + DM as `CNNArchitectureComparison.ipynb` and its siblings (`ActivationComparison.ipynb`, `AdvancedArchitectureComparison.ipynb`, `NormalizationComparison.ipynb`, ...) -- this notebook imports `CNNArchitectureComparison_params.py` directly rather than forking its own copy, so it stays on the same instrument as the rest of `Tutorials/Advanced/`. The shared `TrainParams` dict's own keys (`lrn`, `TrainRunNb`, `ClosedLoopIterations`, `TestRunNb`, `Seed`) are for the closed-loop reconstructor-comparison notebooks; this notebook adds the handful of keys it actually needs (`DenoiserSteps`, `DenoiserLR`, `LossScale`, `BaseChannels`, `Depth`, `NFramesOptions`) onto the same dict rather than introducing a new params file.

`wfs`/`dm` are built fresh and frozen (`.eval()`) -- there's no `TwinCalibrator` fit here, this is a self-contained demo instrument like the rest of the folder. Note the raw WFS detector frame (`Npix = Nres*sampling = 120`) is a different resolution than the pupil-plane OPD (`Nres = 40`) -- the denoiser operates entirely in the `Npix` frame domain; the two are only brought together later via the classical reconstruction matrix.

`NFramesOptions` is the temporal-history comparison this notebook runs: `n_frames=1` is the single-frame baseline (the input frame *is* the frame being denoised), `5` and `10` add 4 and 9 frames of past context respectively. `n_frames=10` costs roughly 10x the WFS-propagation compute of `n_frames=1` per training step -- training all three back to back is noticeably slower than the single-frame notebook this one started from.

In [ ]:
paramfile = 'CNNArchitectureComparison_params.py'

WFSParams = Config.fromfile(paramfile)['WFSParams']
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

# TrainParams as loaded only has keys for the reconstructor-comparison notebooks
# (lrn, TrainRunNb, ClosedLoopIterations, TestRunNb, Seed). Add what this notebook
# needs instead of forking a separate params file.
TrainParams['DenoiserSteps'] = 3000
TrainParams['DenoiserLR'] = 1e-3
TrainParams['LossScale'] = 1e5
TrainParams['BaseChannels'] = 16
TrainParams['Depth'] = 3
TrainParams['NFramesOptions'] = [1, 5, 10]
TrainParams['ClosedLoopSteps'] = 100

PATH = "../../Data/FrameDenoiser/"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

print(f"Raw detector frame size (Npix): {wfs.Npix}x{wfs.Npix}")
print(f"Pupil-plane OPD resolution (Nres): {wfs.Nres}x{wfs.Nres}")

## Building a temporal burst of frames

`wfs(opd, pupil)` returns `frame_with_noise` (since `WFSParams["useNoise"]=True`), and stores the noiseless `frame_no_noise` as a side effect of the very same call. To get `n_frames` consecutive, temporally-correlated frames, `make_frame_stack` below calls `dataset[0], dataset[1], ..., dataset[n_frames-1]` in sequence -- the same indexing convention `Trainer.train()` already uses for its closed-loop iterations, where consecutive indices are wind-evolved continuations of the same atmosphere batch rather than independent draws. `Nphotons`/`RON` are read once (from the batch, or from the `nphotons`/`ron` override arguments) and held fixed for the whole burst, matching `Trainer.train()`'s own convention of treating them as a property of the detector/exposure rather than something that changes frame to frame. The override arguments let the same helper serve both training (random per-burst `Nphotons`/`RON`, drawn from `WFSParams`'s ranges) and the fixed-operating-point evaluation sweep later on.

In [ ]:
def make_frame_stack(dataset, wfs, n_frames, nphotons=None, ron=None):
    """Draws n_frames consecutive noisy WFS frames for the same underlying,
    wind-evolving atmosphere batch (dataset[0..n_frames-1]), the last of which
    is the "current" frame to be denoised and the rest its temporal history.

    Args:
        nphotons, ron: if given, override the batch's own random draw and are
            held fixed for the whole burst (used by the evaluation sweep's
            fixed operating points). If None, the values dataset[0] draws are
            used instead (for training).

    Returns:
        frame_stack (Nphases, n_frames, H, W): noisy frames, oldest first.
        clean_current (Nphases, H, W): frame_no_noise of the current (last) frame.
        opd_current (Nphases, H, W): ground-truth OPD of the current frame.
        nphotons, ron: the burst's (possibly overridden) photon/read-noise conditioning.
    """
    batch = dataset[0]
    if nphotons is None:
        nphotons = batch["nphotons"]
    if ron is None:
        ron = batch["ron"]
    wfs.SetPhotonsAndRON(nphotons, ron)

    frames = []
    for i in range(n_frames):
        if i > 0:
            batch = dataset[i]
        frames.append(wfs(batch["opd"], batch["pupil"]))

    clean_current = wfs.frame_no_noise
    frame_stack = torch.stack(frames, dim=1)  # (Nphases, n_frames, H, W)
    return frame_stack, clean_current, batch["opd"], nphotons, ron


example_stack, example_clean, _, _, _ = make_frame_stack(dataset, wfs, n_frames=5)

imshow_multiple(
    [{"tensor": example_stack[0, i], "title": ("current" if i == 4 else f"t-{4 - i}")} for i in range(5)]
    + [{"tensor": example_clean[0], "title": "frame_no_noise (current)"}]
)
plt.show()

## `FrameDenoiser`: a FiLM-conditioned encoder-decoder

A small U-Net-style encoder-decoder taking a stack of `n_frames` consecutive raw `Npix x Npix` detector frames as input channels (before any per-pupil cropping) and producing one denoised frame, with skip connections. `depth`, `base_channels`, and `n_frames` are the intended growth knobs -- bump any of them to scale the network's spatial or temporal capacity later without touching anything else.

- **Encoder**: `depth` stages of `ConvBlock` (two 3x3 convs, GroupNorm, SiLU) each followed by 2x average-pool downsampling, channel count doubling from `base_channels` each stage. The first stage's input channel count is `n_frames` (not a hardcoded `1`), so it directly consumes the stacked burst.
- **Bottleneck**: one more `ConvBlock`, then **FiLM conditioning** -- a small MLP maps `[log10(Nphotons), RON]` (already per-sample scalars in the batch, held fixed across the burst -- `PhaseDataset` shapes them `(Nphases, 1, 1)`) to a per-channel scale/shift applied to the bottleneck feature map. This is how the network adapts to the detector's noise regime without needing it encoded as extra spatial input channels.
- **Decoder**: mirrored upsampling (bilinear `interpolate` to each skip's exact resolution, robust to any `depth`/`Npix` combination) with skip concatenation, back down to `base_channels`.
- **Output head**: 1x1 conv -> softplus (non-negativity) -> **divide by the spatial sum**, producing a single denoised frame regardless of `n_frames`. This isn't cosmetic: `frame_no_noise`/`frame_with_noise` are always renormalized to sum to 1 (`TorchPropagator.py`), so the denoiser's output has to be too, both to match the training target's scale and to stay consistent with what `GetReconstructedOPD`'s reference-frame subtraction expects downstream.

`n_frames` only changes the first conv's input-channel count, a negligible fraction of the total parameter budget -- all three variants compared below stay well under 1M parameters.

In [ ]:
def _group_norm_groups(channels, max_groups=8):
    """Largest group count in {8,4,2,1} that evenly divides channels, so
    ConvBlock stays valid for any base_channels the growth knobs are set to."""
    for g in (max_groups, 4, 2, 1):
        if channels % g == 0:
            return g
    return 1


class ConvBlock(nn.Module):
    """Two 3x3 convs (GroupNorm + SiLU each) at a fixed channel count."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(_group_norm_groups(out_channels), out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(_group_norm_groups(out_channels), out_channels),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.block(x)


class FiLM(nn.Module):
    """Maps scalar conditioning (log10(Nphotons), RON) to a per-channel
    scale/shift applied to a feature map -- how the denoiser adapts to the
    detector's noise regime without it being encoded as spatial input channels."""

    def __init__(self, cond_dim, num_features, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 2 * num_features),
        )

    def forward(self, x, cond):
        gamma, beta = self.net(cond).chunk(2, dim=-1)
        return x * (1 + gamma[:, :, None, None]) + beta[:, :, None, None]


class FrameDenoiser(nn.Module):
    """FiLM-conditioned encoder-decoder mapping a stack of n_frames consecutive
    noisy raw WFS detector frames (oldest first, last = current) back toward
    the current frame's noiseless counterpart.

    `depth`/`base_channels`/`n_frames` are the intended growth knobs: increase
    any of them to scale the network's spatial or temporal capacity without
    changing anything else about the architecture.
    """

    def __init__(self, n_frames=1, depth=3, base_channels=16, cond_dim=2, film_hidden=32):
        super().__init__()
        self.n_frames = n_frames
        enc_channels = [base_channels * 2 ** i for i in range(depth)]
        bottleneck_channels = enc_channels[-1] * 2

        self.down_blocks = nn.ModuleList([
            ConvBlock(n_frames if i == 0 else enc_channels[i - 1], enc_channels[i])
            for i in range(depth)
        ])
        self.pool = nn.AvgPool2d(2)

        self.bottleneck = ConvBlock(enc_channels[-1], bottleneck_channels)
        self.film = FiLM(cond_dim, bottleneck_channels, hidden=film_hidden)

        self.up_blocks = nn.ModuleList([
            ConvBlock(
                (bottleneck_channels if i == depth - 1 else enc_channels[i + 1]) + enc_channels[i],
                enc_channels[i],
            )
            for i in reversed(range(depth))
        ])

        self.head = nn.Conv2d(enc_channels[0], 1, kernel_size=1)

    def forward(self, frame_stack, nphotons, ron):
        """frame_stack: (Nphases, n_frames, H, W), oldest first."""
        cond = torch.stack([torch.log10(nphotons).flatten(), ron.flatten()], dim=-1)

        h = frame_stack
        skips = []
        for block in self.down_blocks:
            h = block(h)
            skips.append(h)
            h = self.pool(h)

        h = self.film(self.bottleneck(h), cond)

        for block, skip in zip(self.up_blocks, reversed(skips)):
            h = F.interpolate(h, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            h = block(torch.cat([h, skip], dim=1))

        raw = F.softplus(self.head(h))              # (Nphases,1,H,W), non-negative
        frame = raw / raw.sum(dim=(-2, -1), keepdim=True)
        return frame.squeeze(1)                       # (Nphases,H,W), matches wfs frame convention

In [ ]:
denoisers = {
    n: FrameDenoiser(
        n_frames=n, depth=TrainParams['Depth'], base_channels=TrainParams['BaseChannels']
    ).to(device)
    for n in TrainParams['NFramesOptions']
}

for n, model in denoisers.items():
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"n_frames={n:2d} -- trainable parameters: {n_params:,} (budget: 1,000,000)")

## Training

A short hand-rolled loop per `n_frames` variant, not a `Trainer` subclass -- this is a different task (frame-to-frame regression) from `Trainer.train()`'s closed-loop OPD reconstruction, so its modal/DM machinery doesn't apply here. Data generation (`make_frame_stack`, which draws the burst and propagates each frame) is wrapped in `torch.no_grad()` exactly like `Trainer.train()` does; gradients only flow through the denoiser's own forward pass.

Each variant gets its own fresh `AdamW` optimizer and its own checkpoint, and trains for the same `DenoiserSteps` budget -- the same fairness convention `ActivationComparison.ipynb` uses across its variants (independent i.i.d. training draws per variant; only the final evaluation below is paired/seeded for a true apples-to-apples comparison).

`LossScale` exists purely to bring the loss into a numerically convenient range: sum-normalized frames have tiny per-pixel values, so a plain MSE would sit at a tiny, hard-to-read magnitude. This is a numerical convenience only, not a physical rescaling -- the same kind of scaling `LossFunctions.py`'s `Physics_loss` already applies to its own pixel-wise MSE.

Re-running this cell resumes each variant from `Data/FrameDenoiser/FrameDenoiser_N{n}.pth` if it exists and its shape still matches (e.g. after only bumping `DenoiserSteps`) -- a stale checkpoint left over from a different `BaseChannels`/`Depth`/`n_frames` choice is caught and discarded automatically, matching `ActivationComparison.ipynb`'s fallback for the same situation. **Compute note:** `n_frames=10` needs roughly 10x the WFS propagations per step that `n_frames=1` does, so this cell's total run time is dominated by the largest `n_frames` variant.

In [ ]:
loss_trackers = {}

for n, model in denoisers.items():
    checkpoint_path = PATH + f"FrameDenoiser_N{n}.pth"

    try:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"n_frames={n}: loaded existing checkpoint, continuing training")
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by an earlier,
        # differently-sized version of this variant (e.g. after changing BaseChannels,
        # Depth, or this n's own architecture) -- that old checkpoint is no longer
        # compatible and is discarded rather than crashing the run.
        print(f"n_frames={n}: no usable checkpoint found, training from scratch")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['DenoiserLR'], fused=(device == 'cuda'))
    loss_tracker = torch.zeros(TrainParams['DenoiserSteps'], device=device)

    model.train()
    progressBar = tqdm(range(TrainParams['DenoiserSteps']), desc=f"n_frames={n}")

    for step in progressBar:
        with torch.no_grad():
            frame_stack, clean_current, _, nphotons, ron = make_frame_stack(dataset, wfs, n)

        denoised = model(frame_stack, nphotons, ron)
        loss = TrainParams['LossScale'] * F.mse_loss(denoised, clean_current)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        loss_tracker[step] = loss.detach()

        if step % 100 == 1:
            lower = max(0, step - 100)
            progressBar.set_postfix({'Loss': float(loss_tracker[lower:step].mean())})

    torch.save(model.state_dict(), checkpoint_path)
    loss_trackers[n] = loss_tracker

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

fig, ax = plt.subplots(figsize=(8, 5))
for n, tracker in loss_trackers.items():
    ax.plot(smooth(tracker), label=f"n_frames={n}")
ax.set_xlabel("Iteration")
ax.set_ylabel(f"{TrainParams['LossScale']:.0e} x MSE(denoised, frame_no_noise)")
ax.set_title("Denoiser training loss by history length")
ax.legend()
plt.show()

## Visual sanity check

In [ ]:
for n, model in denoisers.items():
    model.eval()
    with torch.no_grad():
        frame_stack, clean_current, _, nphotons, ron = make_frame_stack(dataset, wfs, n)
        denoised = model(frame_stack, nphotons, ron)

    noisy_current = frame_stack[:, -1]
    imshow_multiple([
        {"tensor": noisy_current[0], "title": f"n_frames={n}: noisy (current)"},
        {"tensor": denoised[0], "title": f"n_frames={n}: denoised"},
        {"tensor": clean_current[0], "title": f"n_frames={n}: noiseless (target)"},
        {"tensor": (denoised[0] - clean_current[0]).abs(), "title": f"n_frames={n}: |denoised - clean|"},
    ])
    plt.show()

## Downstream validation: does denoising (and history length) improve wavefront reconstruction?

Frame-domain loss going down is not, by itself, evidence that denoising helps wavefront reconstruction (see `Ideas/12-wfs-frame-denoiser.md`'s caveats). Two evaluations follow: a cheap **single-shot** reconstruction-accuracy check (no DM correction, no time dependence), and a more realistic **closed-loop** rollout (50-100 steps, leaky-integrator correction) matching how reconstructors are actually evaluated elsewhere in this repo (`Trainer.evaluate()`, `CNNArchitectureComparison.ipynb`, `ActivationComparison.ipynb`). Both build on the same classical "Linear" reconstructor calibration below.

In [ ]:
with torch.no_grad():
    M2C = dm.MakeZernikeM2C()
    modes = dm(M2C.T)
    wfs.BuildReconstructionMatrix(modes)

print(f"Interaction matrix calibrated over {modes.shape[0]} Zernike modes")

### Single-shot reconstruction accuracy

This section builds the classical "Linear" reconstructor via `wfs.BuildReconstructionMatrix` over a Zernike modal basis (`dm.MakeZernikeM2C`), the same mechanism `Tutorials/basics/04_TwinCalibrationGroundTruth.ipynb` uses -- no `Trainer`, no closed loop, no DM correction, just single-shot `GetReconstructedOPD` accuracy.

For each of a few fixed `(Nphotons, RON)` operating points spanning the range `WFSParams` trains across, we compare residual wavefront error (nm RMS, over `dataset.pupil`) reconstructed from: the raw noisy current frame, the oracle noiseless current frame, and each `n_frames` variant's denoised output. To make the comparison across `n_frames` values fair, only the *longest* burst (`max(NFramesOptions)`) is drawn per test sample -- every variant (and the two baselines) then reads off the trailing `n_frames`-length suffix of that same shared draw, so every curve is evaluated against identical underlying atmosphere/noise realizations and `n_frames` is the only thing that differs between them. All cases share the same modal truncation (`DMParams["Nmodes"]` Zernikes), so that error component cancels out of the *comparison* even though it caps how good any absolute number can look.

In [ ]:
def reconstruct_opd(frame, wfs, dm, M2C):
    z = wfs.GetReconstructedOPD(frame)
    return dm(z @ M2C.T)


def residual_nm_rms(opd_hat, opd_gt, pupil):
    diff = (opd_hat - opd_gt)[..., pupil.bool()]
    return torch.sqrt(torch.mean(diff ** 2)).item() * 1e9


photon_exponents = [2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]  # log10(Nphotons), spanning WFSParams["Nphotons"]
fixed_ron = 2.0                           # within WFSParams["RON"]
n_test_batches = 10                       # each test sample draws AtmosParams["Nphases"] independent phases

max_n = max(denoisers)

curve_names = ["Raw noisy", "Oracle (noiseless)"] + [f"Denoised (n_frames={n})" for n in denoisers]
curves = {name: [] for name in curve_names}

for model in denoisers.values():
    model.eval()

for exponent in photon_exponents:
    nphotons_val = torch.full((AtmosParams['Nphases'], 1, 1), 10 ** exponent, device=device)
    ron_val = torch.full((AtmosParams['Nphases'], 1, 1), fixed_ron, device=device)

    opd_gt_all = []
    opd_noisy_all, opd_oracle_all = [], []
    opd_denoised_all = {n: [] for n in denoisers}

    torch.manual_seed(TrainParams['Seed'])
    with torch.no_grad():
        for _ in range(n_test_batches):
            full_stack, clean_current, opd_current, _, _ = make_frame_stack(
                dataset, wfs, max_n, nphotons=nphotons_val, ron=ron_val
            )

            opd_gt_all.append(opd_current)
            opd_noisy_all.append(reconstruct_opd(full_stack[:, -1], wfs, dm, M2C))
            opd_oracle_all.append(reconstruct_opd(clean_current, wfs, dm, M2C))

            for n, model in denoisers.items():
                denoised = model(full_stack[:, -n:], nphotons_val, ron_val)
                opd_denoised_all[n].append(reconstruct_opd(denoised, wfs, dm, M2C))

    opd_gt_cat = torch.cat(opd_gt_all)
    curves["Raw noisy"].append(residual_nm_rms(torch.cat(opd_noisy_all), opd_gt_cat, dataset.pupil))
    curves["Oracle (noiseless)"].append(residual_nm_rms(torch.cat(opd_oracle_all), opd_gt_cat, dataset.pupil))
    for n in denoisers:
        curves[f"Denoised (n_frames={n})"].append(
            residual_nm_rms(torch.cat(opd_denoised_all[n]), opd_gt_cat, dataset.pupil)
        )

    print(f"Nphotons=1e{exponent:.1f}: " +
          ", ".join(f"{name}={values[-1]:.1f} nm" for name, values in curves.items()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label, values in curves.items():
    ax.plot(photon_exponents, values, marker='o', label=label)

ax.set_xlabel("log10(Nphotons)")
ax.set_ylabel("Residual wavefront error (nm RMS)")
ax.set_title(f"Single-shot reconstruction accuracy vs. photon count and history length (RON={fixed_ron})")
ax.legend()
plt.show()

### Closed-loop performance (leaky-integrator rollout)

The single-shot metric above never lets the DM correct anything, so it can't show how the loop actually behaves over time -- noise stability, convergence, steady-state residual under an integrator. `run_closed_loop` below mirrors `Trainer.evaluate()`'s structure (`AI4AO/Trainer.py`) almost exactly -- same `residual_opd = opd_gt - opd_reconstructed`, same leaky-integrator gating (`i > n_steps * 0.3`), same `dm(z @ M2C_T)` correction update -- but swaps the neural `phaseReconstructor` for the classical `wfs.GetReconstructedOPD` path, with an optional denoising step in between.

For the `n_frames` history stack, there's no need to redraw a separate burst the way the single-shot section does: a rolling closed loop already produces exactly one new WFS frame per step, so the last `n_frames` of them (padded by repeating the earliest available frame during the first `n_frames-1` steps, while the loop is still warming up anyway) are used directly -- no extra WFS propagations beyond the one every step already needs, regardless of `n_frames`. This makes the closed-loop evaluation's cost independent of history length, unlike training.

Every case below (raw noisy, oracle, and each `n_frames` variant) reseeds with the same `TrainParams['Seed']` immediately before its own rollout. Since every case makes exactly the same sequence of `dataset[i]`/`wfs(...)` calls regardless of which reconstruction path is used, this reproduces bit-identical atmosphere and noise realizations across all of them -- the same pairing trick `ActivationComparison.ipynb` uses before each variant's `trainer.evaluate()` call -- so `n_frames` (and denoising at all) is the only thing that differs between curves.

In [ ]:
@torch.no_grad()
def run_closed_loop(wfs, dm, dataset, M2C, n_steps, nphotons, ron, denoise=None, gain=None, leak=None):
    """Leaky-integrator closed-loop rollout mirroring Trainer.evaluate()'s
    structure, with the classical Linear reconstructor
    (wfs.GetReconstructedOPD -> dm(z @ M2C.T)) in place of a neural
    phaseReconstructor, optionally denoising the WFS frame first.

    denoise: None reconstructs straight from the raw noisy frame; "oracle"
        reconstructs from wfs.frame_no_noise (a noiseless best case, not
        physically realizable); an (n_frames, model) tuple denoises a rolling
        window of the last n_frames WFS frames produced by THIS rollout --
        no extra WFS propagations beyond the one every step already needs.

    Returns residual_opd, stacked (n_steps, Nphases, Nres, Nres) -- the
    closed loop's actual performance metric.
    """
    M2C_T = M2C.T
    Nmodes = M2C.shape[-1]

    batch = dataset[0]
    opd_gt = batch["opd"]
    pupilGT = batch["pupil"]
    gain = gain if gain is not None else batch["loop_gain"]
    leak = leak if leak is not None else batch["loop_leak"]
    wfs.SetPhotonsAndRON(nphotons, ron)

    z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=wfs.device)
    opd_reconstructed = torch.zeros_like(opd_gt)

    history = []
    residuals = []

    for i in range(n_steps):
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed
        noisy = wfs(residual_opd, pupilGT)

        if denoise is None:
            frame = noisy
        elif denoise == "oracle":
            frame = wfs.frame_no_noise
        else:
            n_frames, model = denoise
            history.append(noisy)
            if len(history) > n_frames:
                history.pop(0)
            padded = [history[0]] * (n_frames - len(history)) + history
            frame = model(torch.stack(padded, dim=1), nphotons, ron)

        z_output = wfs.GetReconstructedOPD(frame)

        if i > n_steps * 0.3:
            z_estimated = z_estimated * leak + gain * z_output

        opd_reconstructed = dm(z_estimated @ M2C_T)
        residuals.append(residual_opd)

    return torch.stack(residuals)


def closed_loop_nm_rms(residual_opd_traj, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS), excluding the
    leaky-integrator warm-up window -- same 0.3 convention run_closed_loop
    and Trainer.evaluate() use internally."""
    n_steps = residual_opd_traj.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = residual_opd_traj[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9

In [ ]:
closed_loop_curves = {"Raw noisy": [], "Oracle (noiseless)": []}
closed_loop_curves.update({f"Denoised (n_frames={n})": [] for n in denoisers})

for model in denoisers.values():
    model.eval()

for exponent in photon_exponents:
    nphotons_val = torch.full((AtmosParams['Nphases'], 1, 1), 10 ** exponent, device=device)
    ron_val = torch.full((AtmosParams['Nphases'], 1, 1), fixed_ron, device=device)

    torch.manual_seed(TrainParams['Seed'])
    residual_raw = run_closed_loop(wfs, dm, dataset, M2C, TrainParams['ClosedLoopSteps'], nphotons_val, ron_val, denoise=None)
    closed_loop_curves["Raw noisy"].append(closed_loop_nm_rms(residual_raw, dataset.pupil))

    torch.manual_seed(TrainParams['Seed'])
    residual_oracle = run_closed_loop(wfs, dm, dataset, M2C, TrainParams['ClosedLoopSteps'], nphotons_val, ron_val, denoise="oracle")
    closed_loop_curves["Oracle (noiseless)"].append(closed_loop_nm_rms(residual_oracle, dataset.pupil))

    for n, model in denoisers.items():
        torch.manual_seed(TrainParams['Seed'])
        residual_n = run_closed_loop(wfs, dm, dataset, M2C, TrainParams['ClosedLoopSteps'], nphotons_val, ron_val, denoise=(n, model))
        closed_loop_curves[f"Denoised (n_frames={n})"].append(closed_loop_nm_rms(residual_n, dataset.pupil))

    print(f"Nphotons=1e{exponent:.1f}: " +
          ", ".join(f"{name}={values[-1]:.1f} nm" for name, values in closed_loop_curves.items()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label, values in closed_loop_curves.items():
    ax.plot(photon_exponents, values, marker='o', label=label)

ax.set_xlabel("log10(Nphotons)")
ax.set_ylabel("Steady-state residual wavefront error (nm RMS)")
ax.set_title(f"Closed-loop performance vs. photon count and history length ({TrainParams['ClosedLoopSteps']}-step rollout, RON={fixed_ron})")
ax.legend()
plt.show()

### Closed-loop convergence trajectory

The steady-state numbers above hide *how* the loop gets there. This re-runs the same five cases at the most photon-starved operating point (`photon_exponents[0]`, where denoising should matter most) and plots residual wavefront error at every individual step, not just the post-warm-up average -- showing the loop actually closing (the dashed line marks where the leaky integrator starts correcting) and settling into steady state.

In [ ]:
def per_step_nm_rms(residual_opd_traj, pupil):
    """nm RMS at each individual step (no warm-up exclusion), aggregated over
    the Nphases batch and the pupil -- one scalar per step."""
    pupil_values = residual_opd_traj[..., pupil.bool()]  # (n_steps, Nphases, Npupil)
    return torch.sqrt(torch.mean(pupil_values ** 2, dim=(1, 2))).cpu().numpy() * 1e9


worst_exponent = photon_exponents[0]
nphotons_val = torch.full((AtmosParams['Nphases'], 1, 1), 10 ** worst_exponent, device=device)
ron_val = torch.full((AtmosParams['Nphases'], 1, 1), fixed_ron, device=device)

trajectories = {}

torch.manual_seed(TrainParams['Seed'])
trajectories["Raw noisy"] = run_closed_loop(wfs, dm, dataset, M2C, TrainParams['ClosedLoopSteps'], nphotons_val, ron_val, denoise=None)

torch.manual_seed(TrainParams['Seed'])
trajectories["Oracle (noiseless)"] = run_closed_loop(wfs, dm, dataset, M2C, TrainParams['ClosedLoopSteps'], nphotons_val, ron_val, denoise="oracle")

for n, model in denoisers.items():
    torch.manual_seed(TrainParams['Seed'])
    trajectories[f"Denoised (n_frames={n})"] = run_closed_loop(wfs, dm, dataset, M2C, TrainParams['ClosedLoopSteps'], nphotons_val, ron_val, denoise=(n, model))

fig, ax = plt.subplots(figsize=(8, 5))
for label, traj in trajectories.items():
    ax.semilogy(per_step_nm_rms(traj, dataset.pupil), label=label)
ax.axvline(TrainParams['ClosedLoopSteps'] * 0.3, color='black', linestyle='--', linewidth=1, label="correction starts")
ax.set_xlabel("Step")
ax.set_ylabel("Residual wavefront error (nm RMS)")
ax.set_title(f"Closed-loop convergence at Nphotons=1e{worst_exponent:.1f}, RON={fixed_ron}")
ax.legend()
plt.show()

## Caveats

- **Stay inside the trained `Nphotons`/`RON` range.** FiLM conditioning lets the network adapt within what it was trained on; nothing about this architecture extrapolates gracefully past `WFSParams["Nphotons"]`/`["RON"]`.
- **More history isn't free to train, and isn't guaranteed to help.** Unlike classic multi-frame denoising where the underlying signal is static, here the wavefront also evolves via wind between frames in the burst -- the older a frame, the less correlated it is with the current one. Whether `n_frames=10` beats `5` or `1` depends on how fast `LoopParams["windSpeedVector"]` moves the atmosphere relative to the WFS frame rate; if a larger `n_frames` doesn't win in either comparison above, that's a real result, not a bug. Note this cost asymmetry is training-side only: each *training* step needs `n_frames` WFS propagations to build a fresh burst, but the *closed-loop* evaluation's rolling window is free (one propagation per step regardless of `n_frames`), since it reuses frames the loop already produced.
- **This is supervised, and only possible because this is a simulator.** A real bench cannot produce an on-demand noiseless reference frame the way `PhaseDataset`/`WFS.Propagator` can -- applying a network trained this way to a real instrument means a sim-to-real transfer of the trained weights, not fine-tuning against real noiseless targets.
- Full design rationale and known risks: `Ideas/12-wfs-frame-denoiser.md`.